# Fakeddit dataset - EDA

In [1]:
# !pip install -q kaggle

In [2]:
# !mkdir -p /content/data/fakeddit
# !kaggle datasets download -d vanshikavmittal/fakeddit-dataset -p /content/data/fakeddit --unzip

In [3]:
import polars as pl
from pathlib import Path

In [ ]:
SAMPLES_DIR = Path("/kaggle/input/datasets/vanshikavmittal/fakeddit-dataset/multimodal_only_samples")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_FCT = 0.75


all_data: pl.DataFrame = pl.concat([
    pl.read_csv(SAMPLES_DIR / filename, separator="\t") for filename in SAMPLES_DIR.iterdir()
]).with_columns(
    pl.from_epoch("created_utc", time_unit="s").alias("created_date")
)

In [5]:
import os

cpu_count = os.cpu_count()
print(cpu_count)

4


In [ ]:
import asyncio
import random
import time
from collections import Counter
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from io import BytesIO
from pathlib import Path
from typing import Literal, TypedDict, Union
from urllib.parse import urlparse

import aiohttp
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
from PIL import Image
from tqdm.auto import tqdm


# ============================================================
# Conservative download settings
# ============================================================

MAX_IN_FLIGHT = 256
PER_HOST_LIMIT = 60

REQUEST_JITTER_SECONDS = 0.05

CONNECT_TIMEOUT = 2
READ_TIMEOUT = 2

MAX_RETRIES = 3
BASE_BACKOFF_SECONDS = 0.75
MAX_BACKOFF_SECONDS = 8.0
BACKOFF_FACTOR = 2.0

RETRY_HTTP_STATUSES = {408, 429, 500, 502, 503, }


# ============================================================
# Output / image settings
# ============================================================



IMAGES_PARQUET_PATH = OUTPUT_DIR / "images.parquet"
ERRORS_PARQUET_PATH = OUTPUT_DIR / "fetch_errors.parquet"
SUMMARY_PATH = OUTPUT_DIR / "fetch_summary.txt"

# images are stored as encoded bytes inside the parquet, resized to a fixed
# square so the file stays small and downstream batches are uniform.
IMAGE_SIZE = (512, 512)
IMAGE_FORMAT = "WEBP"
IMAGE_QUALITY = 90

# rows are flushed to disk every PARQUET_BATCH_SIZE images (one row group each),
# so the encoded bytes never all sit in RAM at once.
PARQUET_BATCH_SIZE = 1024
PARQUET_COMPRESSION = "zstd"

IMAGE_SCHEMA = pa.schema([
    ("id", pa.string()),
    ("clean_title", pa.string()),
    ("image", pa.binary()),
    ("6_way_label", pa.int64()),
    ("split", pa.string()),
])


# ============================================================
# User agents for testing
# ============================================================

BASE_USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:151.0) "
    "Gecko/20100101 Firefox/151.0"
)

USER_AGENTS = [
    f"{BASE_USER_AGENT} test-{i}"
    for i in range(1, 100)
]


# ============================================================
# Result types
# ============================================================

class FetchOk(TypedDict):
    status: Literal["ok"]
    image_id: str
    image_url: str
    data: bytes
    content_type: str | None
    user_agent: str


class FetchError(TypedDict):
    status: Literal["error"]
    image_url: str
    exception_type: str
    exception: str
    http_status: int | None
    user_agent: str | None


FetchResult = Union[FetchOk, FetchError]


# ============================================================
# Incremental parquet writer
# ============================================================

class IncrementalParquetWriter:
    """Append rows and flush them to a parquet file in fixed-size batches.

    Safe to call ``add`` from many coroutines at once: a short buffer lock
    guards the staging list, and a separate writer lock serialises the actual
    (thread-offloaded) parquet writes so the underlying ``ParquetWriter`` is
    only ever touched by one task at a time.
    """

    def __init__(
        self,
        path: str | Path,
        schema: pa.Schema,
        batch_size: int,
        compression: str = "zstd",
    ):
        self.path = Path(path)
        self.schema = schema
        self.batch_size = batch_size
        self.compression = compression

        self.buffer: list[dict] = []
        self.buffer_lock = asyncio.Lock()
        self.writer_lock = asyncio.Lock()
        self.writer: pq.ParquetWriter | None = None
        self.rows_written = 0

    async def add(self, row: dict) -> None:
        rows = None

        async with self.buffer_lock:
            self.buffer.append(row)
            if len(self.buffer) >= self.batch_size:
                rows = self.buffer
                self.buffer = []

        if rows is not None:
            await self._write(rows)

    async def _write(self, rows: list[dict]) -> None:
        async with self.writer_lock:
            await asyncio.to_thread(self._write_table, rows)
            self.rows_written += len(rows)

    def _write_table(self, rows: list[dict]) -> None:
        table = pa.Table.from_pylist(rows, schema=self.schema)

        if self.writer is None:
            self.writer = pq.ParquetWriter(
                str(self.path),
                self.schema,
                compression=self.compression,
            )

        self.writer.write_table(table)

    async def close(self) -> None:
        async with self.buffer_lock:
            rows = self.buffer
            self.buffer = []

        if rows:
            await self._write(rows)

        async with self.writer_lock:
            if self.writer is not None:
                await asyncio.to_thread(self.writer.close)
                self.writer = None


# ============================================================
# Helpers
# ============================================================

def resize_and_encode(data: bytes) -> bytes:
    """Validate, resize to ``IMAGE_SIZE`` (bicubic) and re-encode image bytes.

    Raises if PIL can't decode the payload, which is how non-image responses
    (HTML error pages, truncated downloads, ...) get rejected.
    """
    with Image.open(BytesIO(data)) as img:
        img = img.convert("RGB")
        img = img.resize(IMAGE_SIZE, Image.Resampling.BICUBIC)

        out = BytesIO()
        img.save(out, format=IMAGE_FORMAT, quality=IMAGE_QUALITY)

    return out.getvalue()


def exception_text(e: BaseException) -> str:
    return str(e) or repr(e)


def parse_retry_after_seconds(value: str | None) -> float | None:
    if not value:
        return None

    value = value.strip()

    if value.isdigit():
        return float(value)

    try:
        retry_time = parsedate_to_datetime(value)

        if retry_time.tzinfo is None:
            retry_time = retry_time.replace(tzinfo=timezone.utc)

        now = datetime.now(timezone.utc)
        return max(0.0, (retry_time - now).total_seconds())

    except Exception:
        return None


def backoff_seconds(attempt: int, retry_after: str | None = None) -> float:
    retry_after_seconds = parse_retry_after_seconds(retry_after)

    if retry_after_seconds is not None:
        return min(retry_after_seconds, MAX_BACKOFF_SECONDS)

    deterministic = BASE_BACKOFF_SECONDS * (BACKOFF_FACTOR ** attempt)
    jitter = random.uniform(0, REQUEST_JITTER_SECONDS)

    return min(deterministic + jitter, MAX_BACKOFF_SECONDS)


def choose_user_agent() -> str:
    return random.choice(USER_AGENTS)


def make_headers(user_agent: str) -> dict[str, str]:
    return {
        "User-Agent": user_agent,
        "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
        "Connection": "keep-alive",
    }


# ============================================================
# Fetching
# ============================================================

async def fetch_image_bytes(
    session: aiohttp.ClientSession,
    image_id: str,
    url: str,
) -> FetchResult:
    last_exception_type = "UnknownError"
    last_exception = "Unknown error"
    last_http_status: int | None = None
    last_user_agent: str | None = None

    for attempt in range(MAX_RETRIES + 1):
        user_agent = choose_user_agent()
        last_user_agent = user_agent

        try:
            async with session.get(
                url,
                headers=make_headers(user_agent),
            ) as response:
                http_status = response.status
                last_http_status = http_status
                content_type = response.headers.get("Content-Type")

                if http_status >= 400:
                    response.release()

                    last_exception_type = "HTTPStatusError"
                    last_exception = f"HTTP status {http_status}"

                    if http_status in RETRY_HTTP_STATUSES and attempt < MAX_RETRIES:
                        sleep_for = backoff_seconds(
                            attempt=attempt,
                            retry_after=response.headers.get("Retry-After"),
                        )
                        await asyncio.sleep(sleep_for)
                        continue

                    return {
                        "status": "error",
                        "image_url": url,
                        "exception_type": last_exception_type,
                        "exception": last_exception,
                        "http_status": http_status,
                        "user_agent": user_agent,
                    }

                data = await response.read()

                return {
                    "status": "ok",
                    "image_id": image_id,
                    "image_url": url,
                    "data": data,
                    "content_type": content_type,
                    "user_agent": user_agent,
                }

        except asyncio.TimeoutError as e:
            last_exception_type = "TimeoutError"
            last_exception = exception_text(e)
            last_http_status = None

            if attempt < MAX_RETRIES:
                await asyncio.sleep(backoff_seconds(attempt))
                continue

        except aiohttp.ClientError as e:
            last_exception_type = type(e).__name__
            last_exception = exception_text(e)
            last_http_status = None

            if attempt < MAX_RETRIES:
                await asyncio.sleep(backoff_seconds(attempt))
                continue

        except Exception as e:
            last_exception_type = type(e).__name__
            last_exception = exception_text(e)
            last_http_status = None
            break

    return {
        "status": "error",
        "image_url": url,
        "exception_type": last_exception_type,
        "exception": last_exception,
        "http_status": last_http_status,
        "user_agent": last_user_agent,
    }


async def fetch_worker(
    url_queue: asyncio.Queue,
    session: aiohttp.ClientSession,
    writer: IncrementalParquetWriter,
    errors: list[dict],
    exception_counts: Counter,
    http_status_counts: Counter,
    user_agent_counts: Counter,
    pbar: tqdm,
) -> None:
    """Fetch one item, resize + encode it, and stage it for the parquet."""
    while True:
        item = await url_queue.get()

        if item is None:
            url_queue.task_done()
            return

        image_id, clean_title, url, label, split = item

        result = await fetch_image_bytes(
            session=session,
            image_id=image_id,
            url=url,
        )

        user_agent = result.get("user_agent")
        if user_agent:
            user_agent_counts[user_agent] += 1

        if result["status"] == "ok":
            try:
                image_bytes = await asyncio.to_thread(resize_and_encode, result["data"])

                await writer.add({
                    "id": image_id,
                    "clean_title": clean_title,
                    "image": image_bytes,
                    "6_way_label": label,
                    "split": split,
                })

            except Exception as e:
                exception_counts[type(e).__name__] += 1
                http_status_counts[None] += 1

                errors.append({
                    "id": image_id,
                    "image_url": url,
                    "exception_type": type(e).__name__,
                    "exception": exception_text(e),
                    "http_status": None,
                })

        else:
            exception_counts[result["exception_type"]] += 1
            http_status_counts[result["http_status"]] += 1

            errors.append({
                "id": image_id,
                "image_url": url,
                "exception_type": result["exception_type"],
                "exception": result["exception"],
                "http_status": result["http_status"],
            })

        pbar.update(1)
        url_queue.task_done()


# ============================================================
# Metadata
# ============================================================

def write_results_metadata(
    ok_count: int,
    error_count: int,
    exception_counts: Counter,
    http_status_counts: Counter,
    user_agent_counts: Counter,
    errors: list[dict],
    elapsed: float,
) -> None:
    if errors:
        pl.DataFrame(errors).write_parquet(ERRORS_PARQUET_PATH)

    summary_text = [
        f"ok: {ok_count}",
        f"errors: {error_count}",
        f"elapsed_seconds: {elapsed:.2f}",
        "",
        "settings:",
        f"  MAX_IN_FLIGHT: {MAX_IN_FLIGHT}",
        f"  PER_HOST_LIMIT: {PER_HOST_LIMIT}",
        f"  REQUEST_JITTER_SECONDS: {REQUEST_JITTER_SECONDS}",
        f"  MAX_RETRIES: {MAX_RETRIES}",
        f"  BASE_BACKOFF_SECONDS: {BASE_BACKOFF_SECONDS}",
        f"  MAX_BACKOFF_SECONDS: {MAX_BACKOFF_SECONDS}",
        f"  IMAGE_SIZE: {IMAGE_SIZE}",
        f"  IMAGE_FORMAT: {IMAGE_FORMAT}",
        f"  IMAGE_QUALITY: {IMAGE_QUALITY}",
        "",
        "exception_counts:",
    ]

    for key, value in exception_counts.most_common():
        summary_text.append(f"  {key}: {value}")

    summary_text.append("")
    summary_text.append("http_status_counts:")

    for key, value in http_status_counts.most_common():
        summary_text.append(f"  {key}: {value}")

    summary_text.append("")
    summary_text.append("user_agent_counts:")

    for key, value in user_agent_counts.most_common():
        summary_text.append(f"  {key}: {value}")

    SUMMARY_PATH.write_text("\n".join(summary_text), encoding="utf-8")


# ============================================================
# Main API
# ============================================================

async def fetch_all_images(
    items: list[tuple[str, str | None, str, int, str]],
) -> tuple[int, int, Counter, Counter, Counter, float, str]:
    """Fetch, resize and stream ``(id, clean_title, image, 6_way_label, split)``
    rows into a single parquet. Each worker handles fetch + resize + save."""
    exception_counts = Counter()
    http_status_counts = Counter()
    user_agent_counts = Counter()
    errors: list[dict] = []

    writer = IncrementalParquetWriter(
        path=IMAGES_PARQUET_PATH,
        schema=IMAGE_SCHEMA,
        batch_size=PARQUET_BATCH_SIZE,
        compression=PARQUET_COMPRESSION,
    )

    timeout = aiohttp.ClientTimeout(
        total=None,
        connect=CONNECT_TIMEOUT,
        sock_read=READ_TIMEOUT,
    )

    connector = aiohttp.TCPConnector(
        limit=MAX_IN_FLIGHT,
        limit_per_host=PER_HOST_LIMIT,
        ttl_dns_cache=300,
        enable_cleanup_closed=True,
    )

    url_queue = asyncio.Queue(maxsize=MAX_IN_FLIGHT * 2)

    start = time.time()

    async with aiohttp.ClientSession(
        connector=connector,
        timeout=timeout,
        raise_for_status=False,
    ) as session:
        with tqdm(total=len(items)) as pbar:
            workers = [
                asyncio.create_task(
                    fetch_worker(
                        url_queue=url_queue,
                        session=session,
                        writer=writer,
                        errors=errors,
                        exception_counts=exception_counts,
                        http_status_counts=http_status_counts,
                        user_agent_counts=user_agent_counts,
                        pbar=pbar,
                    )
                )
                for _ in range(MAX_IN_FLIGHT)
            ]

            for item in items:
                await url_queue.put(item)

            for _ in workers:
                await url_queue.put(None)

            await url_queue.join()
            await asyncio.gather(*workers)

    await writer.close()

    elapsed = time.time() - start

    ok_count = writer.rows_written
    error_count = len(errors)

    write_results_metadata(
        ok_count=ok_count,
        error_count=error_count,
        exception_counts=exception_counts,
        http_status_counts=http_status_counts,
        user_agent_counts=user_agent_counts,
        errors=errors,
        elapsed=elapsed,
    )

    print(f"Saved pairs: {ok_count}")
    print(f"Errors: {error_count}")
    print(f"Elapsed: {elapsed:.2f}s")
    print(f"Parquet written to: {IMAGES_PARQUET_PATH}")

    return (
        ok_count,
        error_count,
        exception_counts,
        http_status_counts,
        user_agent_counts,
        elapsed,
        str(IMAGES_PARQUET_PATH),
    )

In [7]:
N_ROWS = int(len(all_data) * SAMPLE_FCT)
LABEL_COL = "6_way_label"
SPLIT_FRACTIONS = {"train": 0.6, "val": 0.2, "test": 0.2}
SEED = 42


def add_stratified_split(
    df: pl.DataFrame,
    label_col: str = LABEL_COL,
    fractions: dict[str, float] = SPLIT_FRACTIONS,
    seed: int = SEED,
) -> pl.DataFrame:
    """Add a ``split`` column with a stratified train/val/test assignment.

    Within each ``label_col`` group rows are shuffled and sliced by
    ``fractions``, so every split keeps the label distribution intact.
    """
    train_f = fractions["train"]
    val_f = fractions["val"]

    # shuffled 0-based rank and total size, computed per label group
    rank = pl.int_range(pl.len()).shuffle(seed=seed).over(label_col)
    group_size = pl.len().over(label_col)

    return df.with_columns(
        pl.when(rank < train_f * group_size)
        .then(pl.lit("train"))
        .when(rank < (train_f + val_f) * group_size)
        .then(pl.lit("val"))
        .otherwise(pl.lit("test"))
        .alias("split")
    )


# 1) plain random 220k subset (shuffle, then take N_ROWS)
subset = all_data.filter(pl.col("image_url").is_not_null())
subset = subset.sample(n=min(N_ROWS, subset.height), shuffle=True, seed=SEED)

# 2) stratified 60/20/20 train/val/test split
subset = add_stratified_split(subset)

print(f"subset: {subset.height} rows")
print(
    subset.group_by("split", LABEL_COL)
    .len()
    .sort(["split", LABEL_COL])
)

# subset.select(["id", "clean_title", "image_url", LABEL_COL, "split"]).write_parquet(
#     OUTPUT_DIR / "dataset_splits.parquet"
# )

# (id, clean_title, url, label, split) work items consumed by fetch_all_images
items = list(
    zip(
        subset.get_column("id").cast(pl.Utf8).to_list(),
        subset.get_column("clean_title").to_list(),
        subset.get_column("image_url").to_list(),
        subset.get_column(LABEL_COL).to_list(),
        subset.get_column("split").to_list(),
    )
)

print(f"fetching {len(items)} images")

(
    ok_count,
    error_count,
    exception_counts,
    http_status_counts,
    user_agent_counts,
    elapsed,
    parquet_path,
) = await fetch_all_images(items)


subset: 68 rows
shape: (15, 3)
┌───────┬─────────────┬─────┐
│ split ┆ 6_way_label ┆ len │
│ ---   ┆ ---         ┆ --- │
│ str   ┆ i64         ┆ u32 │
╞═══════╪═════════════╪═════╡
│ test  ┆ 0           ┆ 5   │
│ test  ┆ 2           ┆ 1   │
│ test  ┆ 4           ┆ 4   │
│ train ┆ 0           ┆ 16  │
│ train ┆ 1           ┆ 2   │
│ …     ┆ …           ┆ …   │
│ val   ┆ 1           ┆ 1   │
│ val   ┆ 2           ┆ 2   │
│ val   ┆ 3           ┆ 1   │
│ val   ┆ 4           ┆ 5   │
│ val   ┆ 5           ┆ 1   │
└───────┴─────────────┴─────┘
fetching 68 images


  0%|          | 0/68 [00:00<?, ?it/s]

Saved pairs: 66
Errors: 2
Elapsed: 6.90s
Parquet written to: /kaggle/working/images.parquet


In [8]:
total = ok_count + error_count

print(f"Images attempted: {total}")

if total > 0:
    print(f"Saved pairs: {ok_count} ({ok_count / total:.2%})")
    print(f"Errors: {error_count} ({error_count / total:.2%})")
    print(f"Elapsed: {elapsed:.1f}s")
    print(f"Images/sec: {total / elapsed:.2f}")

print("\nErrors by exception type:")
for exception_type, count in exception_counts.most_common():
    print(f"{exception_type}: {count}")

print("\nHTTP errors by status code:")
for status, count in http_status_counts.most_common():
    print(f"{status}: {count}")


Images attempted: 68
Saved pairs: 66 (97.06%)
Errors: 2 (2.94%)
Elapsed: 6.9s
Images/sec: 9.86

Errors by exception type:
HTTPStatusError: 2

HTTP errors by status code:
404: 1
429: 1


In [9]:
def preview_parquet(path: str | Path, n_rows: int = 5) -> pl.DataFrame:
    """Preview a parquet file: print its shape + schema and show the first rows.

    Scans lazily so it stays fast on large files. Binary columns (e.g. the
    encoded image bytes) are shown as a ``<col>_bytes`` size instead of the raw
    blob. Returns the previewed head as a DataFrame.
    """
    path = Path(path)
    lf = pl.scan_parquet(path)

    schema = lf.collect_schema()
    n_total = lf.select(pl.len()).collect().item()
    head = lf.head(n_rows).collect()

    print(f"{path.name}: {n_total:,} rows x {len(schema)} cols")
    print("schema:")
    for name, dtype in schema.items():
        print(f"  {name}: {dtype}")

    binary_cols = [name for name, dtype in schema.items() if dtype == pl.Binary]
    display_df = head
    for name in binary_cols:
        sizes = [None if b is None else len(b) for b in head.get_column(name).to_list()]
        display_df = display_df.with_columns(pl.Series(f"{name}_bytes", sizes))
    if binary_cols:
        display_df = display_df.drop(binary_cols)

    display(head)
    return head


# example: preview what the fetcher produced (run after the fetch cell)
preview_parquet(IMAGES_PARQUET_PATH)


images.parquet: 66 rows x 5 cols
schema:
  id: String
  clean_title: String
  image: Binary
  6_way_label: Int64
  split: String


id,clean_title,image,6_way_label,split
str,str,binary,i64,str
"""63r5e5""","""wwii vet receives high school …","b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\x03\x03\x03\x03\x04\x03\x03\x04\x05\x08\x05\x05\x04\x04\x05\x0a\x07\x07\x06\x08\x0c\x0a\x0c\x0c\x0b\x0a\x0b\x0b\x0d""…",0,"""test"""
"""b7egfk""","""i have graphite in my fingerti…","b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\x03\x03\x03\x03\x04\x03\x03\x04\x05\x08\x05\x05\x04\x04\x05\x0a\x07\x07\x06\x08\x0c\x0a\x0c\x0c\x0b\x0a\x0b\x0b\x0d""…",0,"""train"""
"""bk4r39""","""smile behind a sad face""","b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\x03\x03\x03\x03\x04\x03\x03\x04\x05\x08\x05\x05\x04\x04\x05\x0a\x07\x07\x06\x08\x0c\x0a\x0c\x0c\x0b\x0a\x0b\x0b\x0d""…",2,"""train"""
"""d468x6""","""this tiny single malt scotch w…","b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\x03\x03\x03\x03\x04\x03\x03\x04\x05\x08\x05\x05\x04\x04\x05\x0a\x07\x07\x06\x08\x0c\x0a\x0c\x0c\x0b\x0a\x0b\x0b\x0d""…",0,"""train"""
"""cb4xax""","""random tank i saw yesterday""","b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\x03\x03\x03\x03\x04\x03\x03\x04\x05\x08\x05\x05\x04\x04\x05\x0a\x07\x07\x06\x08\x0c\x0a\x0c\x0c\x0b\x0a\x0b\x0b\x0d""…",0,"""test"""


id,clean_title,image,6_way_label,split
str,str,binary,i64,str
"""63r5e5""","""wwii vet receives high school …","b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\x03\x03\x03\x03\x04\x03\x03\x04\x05\x08\x05\x05\x04\x04\x05\x0a\x07\x07\x06\x08\x0c\x0a\x0c\x0c\x0b\x0a\x0b\x0b\x0d""…",0,"""test"""
"""b7egfk""","""i have graphite in my fingerti…","b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\x03\x03\x03\x03\x04\x03\x03\x04\x05\x08\x05\x05\x04\x04\x05\x0a\x07\x07\x06\x08\x0c\x0a\x0c\x0c\x0b\x0a\x0b\x0b\x0d""…",0,"""train"""
"""bk4r39""","""smile behind a sad face""","b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\x03\x03\x03\x03\x04\x03\x03\x04\x05\x08\x05\x05\x04\x04\x05\x0a\x07\x07\x06\x08\x0c\x0a\x0c\x0c\x0b\x0a\x0b\x0b\x0d""…",2,"""train"""
"""d468x6""","""this tiny single malt scotch w…","b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\x03\x03\x03\x03\x04\x03\x03\x04\x05\x08\x05\x05\x04\x04\x05\x0a\x07\x07\x06\x08\x0c\x0a\x0c\x0c\x0b\x0a\x0b\x0b\x0d""…",0,"""train"""
"""cb4xax""","""random tank i saw yesterday""","b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x03\x02\x02\x03\x02\x02\x03\x03\x03\x03\x04\x03\x03\x04\x05\x08\x05\x05\x04\x04\x05\x0a\x07\x07\x06\x08\x0c\x0a\x0c\x0c\x0b\x0a\x0b\x0b\x0d""…",0,"""test"""
